# Compare measured scans: GRE reference, EPI without ACDC and EPI with ACDC

Run the cells **from top to bottom**. This notebook reconstructs three measured scanner
acquisitions using the **same readers and navigator correction as the experiment notebook**.
It compares the experimental ACDC result; it does not rerun the optimization.

```text
acquisition_005/
├── gre_reference.dat
├── scan.dat
├── results/
│   └── scan_corr.dat
└── sequences/
    ├── reference_gre.seq
    └── scan.seq
```

`scan.dat` is the EPI acquired without ACDC shimming. `scan_corr.dat` is the EPI acquired
with ACDC shimming on the scanner. Each EPI is reconstructed independently, including
its own navigator correction when navigators are present. **ACDC shimming and navigator
phase correction are different operations.** All three final images are fully reconstructed.

Matrices, FOV and timing are checked against the sequences. The two EPIs can share
`scan.seq` when that is the sequence actually used for both acquisitions. A separate
sequence path can be entered for the ACDC acquisition if needed.

Arrays and figures stay in this notebook. No B0 map, ACDC calibration, ROI, MATLAB,
optimization or waveform export is required.

## 1. Choose the acquisition and results folders

Change `INPUT_DIR` to select another acquisition and `RESULTS_DIR` to locate the measured
ACDC result. The current folder is named `results`; edit this path if you use another name.
Edit `FILES` if filenames differ. `corrected_sequence` must describe the corrected acquisition;
its default is the same `scan.seq` as the acquisition without ACDC.

`RECEIVER_INDEX` chooses the receive coil for phase and navigator diagnostics (Python index
`0` is the first receiver). The reconstructed RSS magnitudes always use **all** receivers.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "Experiment").is_dir() else Path.cwd().parent
EXPERIMENT_DIR = PROJECT_ROOT / "Experiment"
sys.path.insert(0, str(EXPERIMENT_DIR))

from prepare_acdc_inputs import read_gre, read_scan
from read_acdc_sequence import read_sequence

INPUT_DIR = PROJECT_ROOT / "acquisition_005"
RESULTS_DIR = INPUT_DIR / "results"
FILES = {
    "reference": INPUT_DIR / "gre_reference.dat",
    "epi": INPUT_DIR / "scan.dat",
    "epi_corrected": RESULTS_DIR / "scan_corr.dat",
    "reference_sequence": INPUT_DIR / "sequences/reference_gre.seq",
    "epi_sequence": INPUT_DIR / "sequences/scan.seq",
    "corrected_sequence": INPUT_DIR / "sequences/scan.seq",  # Change if the corrected acquisition used another sequence.
}
RECEIVER_INDEX = 0

print("Acquisition folder:", INPUT_DIR)
print("Measured ACDC result:", FILES["epi_corrected"])

## 2. Check files and prepare the figures

These small helpers display tables and images. Array diagnostics use rows and columns;
the final comparison uses physical readout/phase coordinates in millimetres.
No output folder is created.

In [ ]:
def show_table(rows, headings=("Parameter", "Value")):
    lines = ["| " + " | ".join(headings) + " |", "| " + " | ".join(["---"] * len(headings)) + " |"]
    lines += ["| " + " | ".join(str(value) for value in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def show_figure(fig):
    plt.show()
    plt.close(fig)


def image_panel(ax, values, title, cmap="gray", **kwargs):
    artist = ax.imshow(values, cmap=cmap, interpolation="nearest", **kwargs)
    ax.set(title=title, xlabel="Array column", ylabel="Array row")
    return artist


def physical_extent(image, metadata):
    # Each image keeps its own pixel size. The extent includes pixel edges.
    ny, nx = image.shape
    dx, dy = np.asarray(metadata["FOV_m"]) * 1000 / [nx, ny]
    sag, cor, _ = metadata["slice_position_mm"]
    x0 = cor - (nx // 2) * dx
    y0 = -sag - (ny // 2) * dy
    return [x0 - dx / 2, x0 + (nx - 0.5) * dx,
            y0 - dy / 2, y0 + (ny - 0.5) * dy]


for name, path in FILES.items():
    assert path.is_file(), f"Missing {name}: {path}"
for name in ("reference", "epi", "epi_corrected"):
    assert FILES[name].suffix.lower() == ".dat", f"{name} must be raw Siemens Twix data (.dat)."
assert isinstance(RECEIVER_INDEX, int) and not isinstance(RECEIVER_INDEX, bool) and RECEIVER_INDEX >= 0, "RECEIVER_INDEX must be a nonnegative integer."
show_table([(name, path.name) for name, path in FILES.items()])

## 3. Read the acquisition sequences

The GRE has one RF excitation per acquired line. Each EPI acquires a train of lines after
one RF excitation. Sequence checks and trajectories are shown separately for the reference,
the EPI without ACDC and the EPI with ACDC. Their navigator counts can differ when different
sequences were used; the raw records must match each sequence's navigator count.

The EPI trajectories below contain imaging samples. Leading navigators are handled separately.
The colour is time from the first RF centre, with a separate scale for each trajectory.

In [ ]:
reference_acquisition, reference_timing = read_sequence(FILES["reference_sequence"])
epi_acquisition, epi_timing = read_sequence(FILES["epi_sequence"])
corrected_acquisition, corrected_timing = read_sequence(FILES["corrected_sequence"])
assert reference_timing["sequence_kind"] == "gre", "The reference sequence must be mono-echo 2D GRE/FLASH."
assert epi_timing["sequence_kind"] == corrected_timing["sequence_kind"] == "epi", "Both scan sequences must describe EPI acquisitions."
sequence_inputs = [
    ("GRE reference", reference_acquisition, reference_timing),
    ("EPI ACDC off", epi_acquisition, epi_timing),
    ("EPI ACDC on", corrected_acquisition, corrected_timing),
]
show_table([
    ("Matrix [Nx, Ny]", *[timing["matrix_size"] for _, acquisition, timing in sequence_inputs]),
    ("Readout / phase FOV [mm]", *[np.asarray(acquisition["FOV_m"]) * 1000 for _, acquisition, timing in sequence_inputs]),
    ("TE [ms]", *[acquisition["TE_s"] * 1000 for _, acquisition, timing in sequence_inputs]),
    ("TR [ms]", *[acquisition["TR_s"] * 1000 for _, acquisition, timing in sequence_inputs]),
    ("ADC dwell [us]", *[acquisition["dwell_s"] * 1e6 for _, acquisition, timing in sequence_inputs]),
    ("Navigator count", *[timing["navigator_count"] for _, acquisition, timing in sequence_inputs]),
], ("Sequence parameter", "GRE reference", "EPI ACDC off", "EPI ACDC on"))
fig, axes = plt.subplots(1, 3, figsize=(17, 4), layout="constrained")
for ax, (title, acquisition, timing) in zip(axes, sequence_inputs):
    plot = ax.scatter(acquisition["kx_mInv"], acquisition["ky_mInv"], c=acquisition["time_s"] * 1000, s=8, cmap="viridis")
    ax.set(xlabel="kx [cycles/m]", ylabel="ky [cycles/m]", title=title, aspect="equal")
    fig.colorbar(plot, ax=ax, label="Time from first RF centre [ms]")
show_figure(fig)

## 4. Reconstruct the GRE reference

`read_gre` reuses the existing Cartesian Twix reconstruction. It reorders phase rows,
reverses samples marked `REFLECT`, then applies a centred inverse Fourier transform
independently to each receive coil. The final magnitude is combined as
`RSS = sqrt(sum(abs(coil_images)**2, axis=receiver))`.

Raw k-space and complex coil images have axes `[phase row, receiver, readout column]`.
The reference has no EPI navigator correction. Its matrix, FOV, TE, TR, ADC dwell,
readout polarity and acquired line order are checked against `reference_gre.seq`.
The existing reference adapter supports the centred axial geometry used in the experiment.

In [ ]:
gre, reference_diagnostics = read_gre(FILES["reference"], reference_acquisition, return_intermediates=True)
reference_image = np.asarray(gre["magnitude"], float)
reference_metadata = gre["twix_metadata"]
assert RECEIVER_INDEX < reference_metadata["receiver_count"], "Selected receiver is absent from the GRE reference."
assert np.isfinite(reference_image).all() and reference_image.max() > 0, "Reference reconstruction has invalid or zero signal."
show_table([
    ("Image shape [phase, readout]", reference_image.shape),
    ("Receive coils", reference_metadata["receiver_count"]),
    ("Slice centre [patient mm]", reference_metadata["slice_position_mm"]),
])
fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
image_panel(axes[0], np.log1p(np.linalg.norm(reference_diagnostics["kspace"], axis=1)), "Reference: log k-space RSS")
coil_image = reference_diagnostics["coil_images"][:, RECEIVER_INDEX, :]
image_panel(axes[1], np.abs(coil_image), f"Receiver {RECEIVER_INDEX + 1}: magnitude")
plot = image_panel(axes[2], np.angle(coil_image), f"Receiver {RECEIVER_INDEX + 1}: phase [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
fig.colorbar(plot, ax=axes[2])
image_panel(axes[3], reference_image, "GRE reference: combined RSS")
show_figure(fig)

## 5. Reconstruct both EPI acquisitions independently

`read_scan` separates imaging rows from `PHASCOR` navigators, restores reflected readouts,
then reconstructs the image with the same method as the experiment notebook. This function
is called **once for each measured EPI**. A navigator fit from one scan is never reused
for the other scan.

When three navigators are present, their receive-coil-specific phase correction is applied
to reflected imaging rows. With no navigators, that correction is skipped. RF/ADC phases
are already accounted for by the scanner receiver and are not reapplied.

The labels **ACDC off/on refer to the scanner acquisition**. The k-space before/after
panels below refer only to offline navigator phase correction within that acquisition.

In [ ]:
epi_inputs = [
    ("EPI ACDC off", FILES["epi"], epi_acquisition, epi_timing),
    ("EPI ACDC on", FILES["epi_corrected"], corrected_acquisition, corrected_timing),
]
epi_results = []
for label, path, acquisition, timing in epi_inputs:
    image, metadata, diagnostics = read_scan(path, acquisition, return_intermediates=True)
    assert RECEIVER_INDEX < metadata["receiver_count"], f"Selected receiver is absent from {label}."
    assert np.isfinite(image).all() and image.max() > 0, f"{label} reconstruction has invalid or zero signal."
    for key in ("TE_s", "TR_s", "dwell_s"):
        tolerance = 1e-9 if key == "dwell_s" else 1e-6
        assert abs(metadata[key] - acquisition[key]) <= tolerance, f"{label}: Twix and sequence disagree on {key}."
    assert np.allclose(metadata["FOV_m"], acquisition["FOV_m"], rtol=1e-4, atol=1e-6), f"{label}: FOV mismatch."
    assert metadata["navigator_count"] == timing["navigator_count"], f"{label}: navigator count mismatch."
    assert np.array_equal(metadata["polarity"], acquisition["polarity"]), f"{label}: readout polarity mismatch."
    assert metadata["acquired_phase_rows"] == np.argsort(acquisition["line_index"][:, 0]).tolist(), f"{label}: chronological phase-row order mismatch."
    epi_results.append({"label": label, "image": image, "metadata": metadata, "diagnostics": diagnostics})
    print(label, "—", path.name)
    show_table([
        ("Image shape [phase, readout]", image.shape),
        ("Receive coils", metadata["receiver_count"]),
        ("Navigators", metadata["navigator_count"]),
        ("Reflected imaging rows", int(diagnostics["negative_readout_rows"].sum())),
        ("Slice centre [patient mm]", metadata["slice_position_mm"]),
    ])
    raw_kspace = np.log1p(np.linalg.norm(diagnostics["kspace"], axis=1))
    corrected_kspace = np.log1p(np.linalg.norm(diagnostics["corrected_kspace"], axis=1))
    upper = max(raw_kspace.max(), corrected_kspace.max())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    image_panel(axes[0], raw_kspace, label + ": before navigator correction", vmin=0, vmax=upper)
    image_panel(axes[1], corrected_kspace, label + ": after navigator correction", vmin=0, vmax=upper)
    show_figure(fig)

# These two arrays contain the final reconstructed EPI images, including their own navigator correction.
epi_image = epi_results[0]["image"]
corrected_epi_image = epi_results[1]["image"]

## 6. Inspect each scan's navigators and phase fit

The supported three-navigator order is **positive / negative / positive** (`+ / − / +`).
After reflected readouts are reversed, a readout-only inverse Fourier transform places
the navigators in a hybrid domain. The two positive navigators are averaged; their relative
phase to the negative navigator is unwrapped and fitted as an offset plus slope.

The fit uses samples whose weight `abs(negative) * abs(mean_positive)` is above 10% of its
maximum. At least four samples are needed; otherwise the reader leaves the correction at zero.
The saved fit is the phase subtracted from reflected imaging rows. These diagnostics are
shown separately for **ACDC off** and **ACDC on**. A scan without navigators is reported
explicitly and has no navigator figure.

In [ ]:
for result in epi_results:
    label = result["label"]
    metadata = result["metadata"]
    diagnostics = result["diagnostics"]
    if metadata["navigator_count"] == 0:
        print(label + ": no navigator records; no navigator phase correction is applied.")
    else:
        navigator_kspace = diagnostics["navigator_kspace"][:, RECEIVER_INDEX, :]
        navigators = diagnostics["navigator_hybrid"][:, RECEIVER_INDEX, :]
        fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
        image_panel(axes[0], np.log1p(np.abs(navigator_kspace)), label + ": navigator k-space")
        image_panel(axes[1], np.abs(navigators), label + ": navigator hybrid magnitude")
        for ax in axes:
            ax.set(ylabel="Navigator readout", yticks=[0, 1, 2], yticklabels=["1: +", "2: −", "3: +"])
        show_figure(fig)

        positive = (navigators[0] + navigators[2]) / 2
        relative_phase = np.unwrap(np.angle(navigators[1] * positive.conj()))
        weight = np.abs(navigators[1]) * np.abs(positive)
        fit_samples = weight > 0.10 * weight.max()
        fitted_phase = diagnostics["navigator_phase_correction_rad"][RECEIVER_INDEX]
        position = np.linspace(-1, 1, navigators.shape[1])
        fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
        for index, navigator_label in enumerate(("1: +", "2: −", "3: +")):
            axes[0].plot(position, np.abs(navigators[index]), label=navigator_label)
        axes[0].set(xlabel="Normalized readout position", ylabel="Magnitude", title=label + ": navigator magnitudes")
        axes[0].legend()
        axes[1].plot(position, relative_phase, ".-", color="gray", label="Measured unwrapped phase")
        axes[1].scatter(position[fit_samples], relative_phase[fit_samples], color="tab:blue", label="Above weight threshold")
        axes[1].plot(position, fitted_phase, color="tab:red", label="Saved affine correction")
        axes[1].set(xlabel="Normalized readout position", ylabel="Phase [rad]", title=label + ": negative versus mean positive")
        axes[1].legend()
        plot = image_panel(axes[2], diagnostics["navigator_phase_correction_rad"], label + ": saved correction [rad]", cmap="RdBu_r")
        axes[2].set(ylabel="Receiver index", xlabel="Readout column")
        fig.colorbar(plot, ax=axes[2])
        show_figure(fig)
        print(f"{label}, receiver {RECEIVER_INDEX + 1}: {int(fit_samples.sum())} samples pass the fit-weight threshold.")
        if fit_samples.sum() < 4:
            print("Fewer than four reliable samples: this receiver keeps a zero correction.")

## 7. Inspect navigator correction within each EPI acquisition

For each scan, the before/after RSS images share one intensity scale. Their only difference
is the offline navigator phase correction. These panels show how navigator processing affects
that scan; they are **not** the comparison of ACDC off versus ACDC on.

The difference panel has its own colour scale. It is not a measure of shimming performance.
The comparison of the measured ACDC result is the three-image figure in the next step.

In [ ]:
for result in epi_results:
    label = result["label"]
    before = result["diagnostics"]["uncorrected_image"]
    after = result["image"]
    upper = max(before.max(), after.max())
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
    image_panel(axes[0], before, label + ": before navigator correction", vmin=0, vmax=upper)
    image_panel(axes[1], after, label + ": after navigator correction", vmin=0, vmax=upper)
    plot = image_panel(axes[2], np.abs(after - before), label + ": absolute RSS change", cmap="magma")
    fig.colorbar(plot, ax=axes[2])
    show_figure(fig)

## 8. Compare the measured ACDC result

The three panels are **GRE reference**, **EPI without ACDC** (`scan.dat`) and
**EPI with ACDC** (`results/scan_corr.dat`). Both EPI images include their own navigator
correction, when present. The ACDC effect therefore compares two actual scanner acquisitions.

The two EPIs use **one common intensity scale**, set by their shared maximum, so their
relative signal amplitudes are preserved. The GRE uses its own maximum because its TE,
TR and contrast differ. This display scaling does not change the reconstructed arrays.
A GRE-versus-EPI MSE is not computed.

Each scan keeps its own matrix, FOV and pixel size. Physical axes are used when all slice
centres and orientations match the supported axial frame. Otherwise the images are displayed
in array coordinates with a message. No image interpolation or registration is performed.

In [ ]:
comparison = [
    ("GRE reference", reference_image, reference_metadata),
    ("EPI without ACDC", epi_image, epi_results[0]["metadata"]),
    ("EPI with ACDC", corrected_epi_image, epi_results[1]["metadata"]),
]
geometry_checks = []
for title, image, metadata in comparison:
    same_centre = np.allclose(reference_metadata["slice_position_mm"], metadata["slice_position_mm"], rtol=0, atol=1e-3)
    same_orientation = np.isclose(abs(np.dot(reference_metadata["slice_quaternion_wxyz"], metadata["slice_quaternion_wxyz"])), 1, rtol=0, atol=1e-8)
    geometry_checks.append((title, bool(same_centre), bool(same_orientation), image.shape))
show_table(geometry_checks, ("Acquisition", "Same centre as reference", "Same orientation", "Image shape"))
physical_view = all(centre and orientation for _, centre, orientation, shape in geometry_checks)

common_epi_scale = max(epi_image.max(), corrected_epi_image.max())
display_scales = [reference_image.max(), common_epi_scale, common_epi_scale]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), layout="constrained")
for ax, (title, image, metadata), scale in zip(axes, comparison, display_scales):
    displayed = image / scale
    if physical_view:
        plot = ax.imshow(displayed, cmap="gray", vmin=0, vmax=1, interpolation="nearest",
                         origin="lower", extent=physical_extent(image, metadata), aspect="equal")
        ax.set(xlabel="X: readout [mm]", ylabel="Y: phase [mm]", title=title)
    else:
        plot = image_panel(ax, displayed, title + " (array coordinates)", vmin=0, vmax=1)
    scale_label = "RSS / GRE maximum" if title == "GRE reference" else "RSS / common EPI maximum"
    fig.colorbar(plot, ax=ax, label=scale_label)
show_figure(fig)
if not physical_view:
    print("Geometry differs: the three scans are shown in array coordinates and are not registered.")
print("Reconstruction complete. Arrays: reference_image, epi_image (ACDC off), corrected_epi_image (ACDC on).")

## Reading the figures

1. Verify the parameters and trajectories for the reference and both EPI acquisitions.
2. Inspect GRE k-space, receive-coil magnitude/phase and combined RSS.
3. Inspect the navigators and fitted phase independently for ACDC off and ACDC on.
4. Check navigator correction before/after within each EPI acquisition.
5. Use the final three-image figure to compare the scanner result with and without ACDC.

Save the notebook to keep its figures and parameters. Change the input/results folders,
sequence paths or selected receiver, then rerun the relevant cells. No pipeline output
folder is created and no scanner or FPGA is controlled.